# Bigger backbones on a Colab GPU (DenseNet121 · ConvNeXt-Tiny · EfficientNet-B2)

Run on **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all** (~1–1.5 h).

Protocol (same as the shipped v3 model, see README):
1. Train each backbone on the Kermany training set (patient-level validation split, histogram equalisation).
2. Adapt each one to the test-set source using only the 30% **calibration** patients (`reports/test_split.json`), with 3-fold CV inside them.
3. **The winner is chosen by out-of-fold calibration accuracy only.** Holdout numbers are printed for information, not used for the choice.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!rm -rf repo && git clone -q -b colab-bigger-models https://github.com/sachinu25/Chest-X-Ray-Pneumonia-Detection.git repo
%cd /content/repo
!pip -q install pyarrow
import torch, torchvision; print(torch.__version__, torchvision.__version__, torch.cuda.get_device_name(0))

In [ ]:
import os, subprocess
os.makedirs('/content/data', exist_ok=True)
base = 'https://huggingface.co/datasets/hf-vision/chest-xray-pneumonia/resolve/main/data/'
files = {f'train-0000{i}-of-00007.parquet': f'train-0000{i}-of-00007.parquet' for i in range(7)}
files['validation-00000-of-00001.parquet'] = 'validation-00000-of-00001.parquet'
files['test-00000-of-00001.parquet'] = 'test.parquet'
for src, dst in files.items():
    if not os.path.exists(f'/content/data/{dst}'):
        subprocess.run(['wget', '-q', '-O', f'/content/data/{dst}', base + src], check=True)
!ls -la /content/data

In [ ]:
ARCHS = ['densenet121', 'convnext_tiny', 'efficientnet_b2']
import os
for arch in ARCHS:
    for attempt in range(3):
        if os.path.exists(f'/content/candidates/{arch}.json'):
            break
        !python -u scripts/train_v2.py --data-dir /content/data --archs {arch} --epochs 8 --size 224 --batch 32 --cand-dir /content/candidates 2>&1 | grep -v Warning | grep -E "epoch|val @thr|Error|resumed"
!ls /content/candidates

In [ ]:
for arch in ARCHS:
    if os.path.exists(f'/content/candidates/{arch}.pth') and not os.path.exists(f'/content/adapted/adaptation_{arch}.json'):
        !python -u scripts/adapt_target.py --data-dir /content/data --base /content/candidates/{arch}.pth --arch {arch} --out-dir /content/adapted 2>&1 | grep -v Warning | grep -E "calibration|^CV|chosen|FINAL|accuracy|roc_auc|Error"


In [ ]:
import json, glob
rows = []
# shipped v3 ResNet18 (reports/adaptation.json in the repo) for comparison
base = json.load(open('reports/adaptation.json'))['final']
rows.append(('resnet18 (shipped v3)', base['oof_calibration']['accuracy'], base['oof_calibration']['roc_auc'],
             base['holdout']['accuracy'], base['holdout']['roc_auc']))
for f in sorted(glob.glob('/content/adapted/adaptation_*.json')):
    r = json.load(open(f)); fin = r['final']
    rows.append((r['arch'], fin['oof_calibration']['accuracy'], fin['oof_calibration']['roc_auc'],
                 fin['holdout']['accuracy'], fin['holdout']['roc_auc']))
print(f"{'model':24s} {'OOF calib acc':>13s} {'OOF AUC':>8s} | {'holdout acc':>11s} {'holdout AUC':>11s}")
for r in rows:
    print(f"{r[0]:24s} {r[1]:13.4f} {r[2]:8.4f} | {r[3]:11.4f} {r[4]:11.4f}")
winner = max(rows, key=lambda r: (r[1], r[2]))
print()
print('WINNER (chosen by OOF calibration accuracy, then AUC):', winner[0])

In [ ]:
import shutil
os.makedirs('/content/export', exist_ok=True)
for f in glob.glob('/content/adapted/*'):
    shutil.copy(f, '/content/export/')
shutil.make_archive('/content/xray_colab_results', 'zip', '/content/export')
from google.colab import files
files.download('/content/xray_colab_results.zip')